# 净利润断层：业绩参数回测

## tl;dr

在保持线上技术条件（高开 ≥ 1.5%、成交额量比 ≥ 1.3、成交额 ≥ 3000 万、上市 ≥ 120 日、收阳且未封板）不变时，稳健候选为：净利同比 80%～1000%、营收同比 ≥ 20%、净利环比 0%～1000%、营收环比仅设 ≤ 100%。2018-08 至 2026-08 的完整 20 日样本中，20 日平均收益由 2.11% 提升到 3.74%，平均超额由 2.08% 提升到 3.77%，中位收益由 0.49% 提升到 1.15%，样本由 2535 条降到 757 条。月度簇自助法估计的超额提升 95% 区间为约 0.60～2.89 个百分点。

结论使用“按事件源能力生效”的口径：净利/营收环比只过滤财报，营收同比过滤财报和快报，预告不因缺字段被淘汰。

## Context & Methods

- 决策：为净利润断层模块选择不依赖单一历史时期、且有财务含义的默认业绩阈值。
- 数据：生产 DuckDB 分析库，只读；财报、快报、预告、日 K、复权因子及沪深 300。
- 事件口径：每个股票/报告期/事件源取首次公告；公告后的首个交易日为 T+1；同一报告期多个事件触发时留最早一条。
- 收益口径：T+1 收盘买入，前复权计算 5/20/90 个交易日收益；超额相对同期沪深 300。
- 防过拟合：参数先做单变量扫描，再比较少量具备财务含义的组合；分为 2018–2022 训练、2023–2024 验证、2025–2026 测试；最后按信号月份做簇自助法。
- 近期样本的 90 日收益可能尚未成熟，参数选择以 20 日收益/超额为主，5 日与 90 日作为护栏。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == 'lab':
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'lab'))

from earnings_gap_parameter_analysis import (
    LIVE_GROWTH_CONFIG, RECOMMENDED_GROWTH_CONFIG,
    apply_growth_config, build_signal_frame, compare_configs,
    monthly_cluster_bootstrap, summarize,
)

plt.rcParams.update({
    'figure.figsize': (10, 5.5), 'figure.dpi': 120,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'axes.axisbelow': True,
    'grid.alpha': 0.22, 'font.size': 10,
})
pd.set_option('display.max_columns', 40)

## Data

### 1. 重建线上技术条件下的历史候选

In [ ]:
signals, metadata = build_signal_frame()
metadata

### 2. 数据质量与字段覆盖

新增三项指标在财报信号中的覆盖率接近 100%，但历史快报的 `yoy_sales` 几乎全空，预告本身也不提供这些字段。因此不能把缺失直接当作不达标，否则一旦设置新增阈值，事件源结构会从财报/预告/快报退化成几乎只有财报。

In [ ]:
coverage = (
    signals.groupby('source')[['or_yoy', 'np_qoq', 'or_qoq']]
    .agg(lambda values: values.notna().mean() * 100)
    .rename(columns={'or_yoy': '营收同比覆盖%', 'np_qoq': '净利环比覆盖%', 'or_qoq': '营收环比覆盖%'})
)
freshness = pd.Series(metadata['freshness'], name='最新日期').to_frame()
display(coverage.round(2))
display(freshness)

## Results

### 3. 单变量阈值扫描

每条曲线只改一个阈值，其余沿用线上净利同比 25%～3000%。图中同时标注完整 20 日样本量，避免只看均值。

In [ ]:
scan_specs = {
    'Min profit YoY (%)': ('min_profit_yoy', [0, 20, 25, 30, 50, 80, 100, 150, 200]),
    'Min revenue YoY (%)': ('min_revenue_yoy', [None, 0, 10, 20, 30, 50]),
    'Min profit QoQ (%)': ('min_profit_qoq', [None, -20, 0, 20, 50, 100]),
    'Max revenue QoQ (%)': ('max_revenue_qoq', [None, 50, 100, 200, 500]),
}
scan_rows = []
for panel, (key, values) in scan_specs.items():
    for value in values:
        config = {**LIVE_GROWTH_CONFIG, key: value}
        result = summarize(apply_growth_config(signals, config))
        scan_rows.append({
            'panel': panel, 'value': 'None' if value is None else str(value),
            '20d excess (%)': result['exc20_mean_pct'], 'n20': result['n20'],
        })
scan = pd.DataFrame(scan_rows)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for axis, (panel, part) in zip(axes.flat, scan.groupby('panel', sort=False)):
    x = np.arange(len(part))
    axis.plot(x, part['20d excess (%)'], color='#315E8A', marker='o', linewidth=2)
    axis.set_xticks(x, part['value'])
    axis.set_title(panel)
    axis.set_ylabel('20-day excess return (%)')
    for xi, yi, count in zip(x, part['20d excess (%)'], part['n20']):
        axis.annotate(f'n={int(count)}', (xi, yi), xytext=(0, 7), textcoords='offset points',
                      ha='center', fontsize=8, color='#555555')
plt.show()
scan.round(2)

### 4. 有财务含义的组合与时间外验证

推荐候选没有取网格中的最高历史收益点，而选择参数平台内更简单、样本更多的一组：净利同比 80%～1000%、营收同比 ≥ 20%、净利环比 0%～1000%、营收环比 ≤ 100%。营收环比不设下限，因为季节性使一季报/年报的环比不可直接横比；只用上限剔除异常基数。

In [ ]:
configs = {
    '线上现配置': LIVE_GROWTH_CONFIG,
    '温和质量': {
        'min_profit_yoy': 50, 'max_profit_yoy': 1000, 'min_revenue_yoy': 0,
    },
    '稳健推荐': RECOMMENDED_GROWTH_CONFIG,
    '更激进': {**RECOMMENDED_GROWTH_CONFIG, 'min_profit_yoy': 100},
}
comparison = compare_configs(signals, configs)
comparison_columns = [
    '事件源', 'n20', 'ret5_mean_pct', 'ret20_mean_pct', 'ret20_median_pct',
    'ret20_win_pct', 'exc20_mean_pct', 'ret90_mean_pct',
    '训练2018-2022样本', '训练2018-2022超额%',
    '验证2023-2024样本', '验证2023-2024超额%',
    '测试2025-2026样本', '测试2025-2026超额%',
]
comparison[comparison_columns].round(2)

In [ ]:
period_columns = ['训练2018-2022超额%', '验证2023-2024超额%', '测试2025-2026超额%']
plot_data = comparison.loc[['线上现配置', '稳健推荐'], period_columns].T
plot_data.index = ['Train 2018-22', 'Validation 2023-24', 'Test 2025-26']
axis = plot_data.plot.bar(color=['#9BA7B4', '#315E8A'], width=0.72, figsize=(10, 5.5))
axis.set_title('20-day excess return remains positive across time splits')
axis.set_ylabel('Mean excess return (%)')
axis.set_xlabel('')
axis.axhline(0, color='#333333', linewidth=0.8)
axis.legend(['Current', 'Recommended'], frameon=False, loc='upper left')
axis.tick_params(axis='x', rotation=0)
for container in axis.containers:
    axis.bar_label(container, fmt='%.2f', padding=3)
plt.show()

### 5. 不确定性：按月份成簇抽样

同一财报季的信号并不独立，因此不对单条股票信号做普通 IID 自助法，而按月份整体重抽 10,000 次。区间描述历史样本对月份变化的敏感度，不等同于未来收益保证。

In [ ]:
bootstrap = monthly_cluster_bootstrap(
    signals, LIVE_GROWTH_CONFIG, RECOMMENDED_GROWTH_CONFIG, draws=10_000,
)
bootstrap.round(2)

## Takeaways

1. **推荐默认值**：净利同比 80%～1000%；净利环比 0%～1000%；营收同比下限 20%、上限留空；营收环比下限留空、上限 100%。
2. **主要收益来自质量约束而非极端网格点**：营收同比下限提升最稳定；净利环比为正进一步改善验证期表现；净利/营收环比上限用于剔除小基数异常。
3. **必须按事件源能力应用过滤**：预告不提供三项新增指标，不应因字段缺失被淘汰；历史快报营收同比覆盖接近 0%，设置营收同比后快报会自然退出。
4. **稳健但非无风险**：推荐参数在 2019 年的 20 日收益/超额仍为负；2026 年 90 日样本尚未成熟；历史涨停价按板块/ST 规则估算；未计手续费与滑点（两方案同口径，加入成本会同时下移）。
5. **解释边界**：结果支持把推荐值作为默认筛选器，不支持把 3.74% 当作可实现的组合收益；信号会重叠，实际资金占用、仓位和交易冲击需要单独做组合级回测。